# Member 3 — Classification Preprocessing, Models, Evaluation

Dataset: `data/adult/adult_train.csv` / `adult_test.csv` (UCI Adult / Census Income).
Target: `income` (`<=50K` / `>50K`), binary classification.
Scope: preprocessing, Logistic Regression, KNN, Gaussian Naive Bayes, Decision Tree, SVC,
evaluation (accuracy, weighted F1, confusion matrix per algorithm).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

RANDOM_STATE = 42
sns.set_style('whitegrid')
np.random.seed(RANDOM_STATE)

## 1. Load raw train/test split

The Adult dataset ships with a pre-defined train/test split (`adult_train.csv`, `adult_test.csv`),
so no re-splitting is needed. Missing values are encoded as `'?'` in categorical columns.

In [2]:
TRAIN_PATH = Path('../data/adult/adult_train.csv')
TEST_PATH = Path('../data/adult/adult_test.csv')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
train_df.head()

train shape: (32561, 15)
test shape: (16281, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Clean missing values

`'?'` marks missing entries in `workclass`, `occupation`, `native-country`. Rows with any
missing value are dropped (small fraction of the data) rather than imputed, to keep the
classification signal clean.

In [3]:
for df in (train_df, test_df):
    df.replace('?', np.nan, inplace=True)

print('train missing per column:')
print(train_df.isna().sum()[train_df.isna().sum() > 0])

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print('\nafter dropna -> train:', train_df.shape, 'test:', test_df.shape)

train missing per column:
workclass         1836
occupation        1843
native-country     583
dtype: int64

after dropna -> train: (30162, 15) test: (15060, 15)


## 3. Encode categorical features + target

Categorical predictors are label-encoded (tree/distance models handle this fine here; a
one-hot alternative would blow up dimensionality for `native-country`). Encoders are fit on
train only and reused on test to avoid leakage / unseen-category mismatches.

In [4]:
target_col = 'income'
cat_cols = train_df.select_dtypes(include='object').columns.tolist()
cat_cols.remove(target_col)

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = test_df[col].map(lambda v, le=le: le.transform([v])[0] if v in le.classes_ else -1)
    encoders[col] = le

target_le = LabelEncoder()
train_df[target_col] = target_le.fit_transform(train_df[target_col])
test_df[target_col] = target_le.transform(test_df[target_col])
print('target classes:', dict(zip(target_le.classes_, target_le.transform(target_le.classes_))))

# unseen categories only ever land in the small dropped/edge set; drop any -1 leftovers
test_df = test_df[(test_df[cat_cols] != -1).all(axis=1)].reset_index(drop=True)
print('test shape after unseen-category filter:', test_df.shape)

target classes: {'<=50K': np.int64(0), '>50K': np.int64(1)}
test shape after unseen-category filter: (15060, 15)


## 4. Feature/target split + scaling

`StandardScaler` is fit on train only, then applied to test — needed for KNN and SVC which
are distance-based; harmless for Logistic Regression, GaussianNB, and the Decision Tree.

In [5]:
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print('X_train:', X_train_scaled.shape, ' X_test:', X_test_scaled.shape)
print('class balance (train):')
print(y_train.value_counts(normalize=True))

X_train: (30162, 14)  X_test: (15060, 14)
class balance (train):
income
0    0.751078
1    0.248922
Name: proportion, dtype: float64
